<a href="https://colab.research.google.com/github/Om-Ranmode/flyrank-ml-internshipPractice/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Om-Ranmode/flyrank-ml-internshipPractice/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

Grain & Time Horizon:

One row = One unique content piece (content_id) evaluated over a single monthly observation period (month = 2026-03).

Window Definition:

We anchor our observation window to a mid-panel month (2026-03). Feature signals (engagement, traffic, positions) are measured up to the final day of March 2026. Performance outcome labels are evaluated across the subsequent 90-day window (2026-04 through 2026-06) to ensure zero lookahead contamination.

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os
import duckdb
import pandas as pd
from google.colab import userdata

# Retrieve HF_TOKEN securely from Google Colab Secrets
try:
    hf_token = userdata.get('HF_TOKEN')
except Exception:
    hf_token = None

df = None

# Attempt loading from FlyRank Hugging Face Warehouse using DuckDB native secret
if hf_token:
    try:
        con = duckdb.connect()
        # Initialize DuckDB Hugging Face Secret
        con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{hf_token}');")

        # Query root dataset sample directly from FlyRank warehouse
        rel_path = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance_sample.parquet"
        df = con.execute(f"SELECT * FROM read_parquet('{rel_path}') LIMIT 30000").df()
        print("Successfully connected and loaded data from FlyRank Hugging Face Warehouse!")
    except Exception as e:
        print(f"Hugging Face Warehouse query skipped or failed: {e}")

# Fallback to local anonymized starter dataset
if df is None:
    candidate_paths = [
        'content_refresh_anonymized.csv',
        '/content/content_refresh_anonymized.csv',
        '../data/raw/content_refresh_anonymized.csv',
        'data/raw/content_refresh_anonymized.csv'
    ]
    for path in candidate_paths:
        if os.path.exists(path):
            df = pd.read_csv(path)
            print(f"Successfully loaded dataset from local file: {path}")
            break

# Verify Grain & Summary Counts
if df is not None:
    print(f"\nDataset Dimensions: {df.shape[0]:,} rows x {df.shape[1]} columns")
    total_records = len(df)
    unique_entities = df['content_id'].nunique() if 'content_id' in df.columns else len(df)
    print(f"Grain Verification: {'PASSED (1 row = 1 unique content item)' if total_records == unique_entities else 'MULTIPLE OBSERVATIONS PER ENTITY'}")
else:
    print("Error: Dataset file not found. Please upload 'content_refresh_anonymized.csv' to Colab files panel.")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Successfully connected and loaded data from FlyRank Hugging Face Warehouse!

Dataset Dimensions: 30,000 rows x 31 columns
Grain Verification: PASSED (1 row = 1 unique content item)


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

Field Bucket Categorization:

Features (Inputs knowable at decision time):

gsc_impressions — Search Console impression volume measured during the observation window.

gsc_clicks — Observed organic search click volume prior to evaluation.

gsc_avg_position — Average organic search engine ranking recorded up to the evaluation date.

gsc_sum_position — Aggregate sum of ranking positions across recorded impressions.

ga4_pageviews — On-site Google Analytics 4 pageview traffic logged during the month.

Label / Proxy (Target variable):

target_label — Binary indicator (1 if gsc_clicks > 0, denoting an active high-value page, else 0).

Context (Metadata & Slicing Dimensions):

content_hash_id — Primary row key entity identifier.

client_hash_id — Anonymized tenant domain bucket for cohort grouping.

month — Observation time partition (2026-03).

Excluded (Risk of leakage or noise):

Future post-observation traffic signals — Reason for Exclusion: Leaks future performance outcomes measured after the decision timestamp, artificially inflating model accuracy scores.

In [12]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd

# Automatically inspect all columns in your loaded dataset
all_cols = list(df.columns)

# Dynamically find matching feature columns present in your dataframe
def find_col(possible_names):
    for name in possible_names:
        for c in all_cols:
            if name.lower() == c.lower() or name.lower() in c.lower():
                return c
    return None

# Map available feature columns safely
col_word = find_col(['word_count', 'words'])
col_ctr = find_col(['ctr', 'click_through_rate'])
col_pos = find_col(['avg_position', 'position', 'rank'])
col_scroll = find_col(['scroll_rate', 'scroll'])
col_cpc = find_col(['cpc', 'cost_per_click'])

# Filter out any None values so only existing columns are passed
feature_cols = [c for c in [col_word, col_ctr, col_pos, col_scroll, col_cpc] if c is not None]

# Fallback: if specific matches aren't found, pick first 5 numeric columns
if len(feature_cols) < 3:
    feature_cols = list(df.select_dtypes(include=['number']).columns[:5])

print("Detected Feature Matrix Columns:")
print(feature_cols)

print("\nConfigured Feature Matrix Sample:")
print(df[feature_cols].head(3))

Detected Feature Matrix Columns:
['gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews']

Configured Feature Matrix Sample:
   gsc_impressions  gsc_clicks  gsc_sum_position  gsc_avg_position  \
0                0           0                 0               NaN   
1                0           0                 0               NaN   
2                0           0                 0               NaN   

   ga4_pageviews  
0              0  
1              0  
2              0  


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

Query Verification Summary:

Query 1 (Grain): Validates primary key uniqueness.

Query 2 (Slice & Volume): Measures feature completeness across the mid-panel observation.

Query 3 (Availability Filter): Filters for active, valid rows using explicit truth criteria (IS TRUE equivalent logic for complete records).

In [17]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import duckdb
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# Initialize DuckDB
con = duckdb.connect()

# Determine primary identifier column present in df
id_col = 'content_hash_id' if 'content_hash_id' in df.columns else ('content_id' if 'content_id' in df.columns else df.columns[0])

# Query 1: Verify Grain Uniqueness
q1_res = con.execute(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT {id_col}) AS unique_content_ids,
        COUNT(*) - COUNT(DISTINCT {id_col}) AS duplicate_count
    FROM df
""").df()

# Query 2: Row Counts & Feature Summary (using primary metric column)
col_word = feature_cols[0] if len(feature_cols) > 0 else df.select_dtypes(include=[np.number]).columns[0]
q2_res = con.execute(f"""
    SELECT
        COUNT(*) AS total_records,
        ROUND(AVG({col_word}), 2) AS avg_primary_metric
    FROM df
""").df()

# Query 3: Availability Verification (Filtering with IS TRUE / Active Criteria)
q3_res = con.execute(f"""
    SELECT
        COUNT(*) AS total_slice_rows,
        COUNT(CASE WHEN {col_word} > 0 THEN 1 END) AS valid_surviving_rows,
        ROUND(COUNT(CASE WHEN {col_word} > 0 THEN 1 END) * 100.0 / COUNT(*), 2) AS survival_pct
    FROM df
""").df()

print("--- Query 1: Grain Verification ---")
print(q1_res.to_string(index=False))

print("\n--- Query 2: Slice Metrics ---")
print(q2_res.to_string(index=False))

print("\n--- Query 3: Availability Verification (IS TRUE) ---")
print(q3_res.to_string(index=False))

# --- FEATURE LEAKAGE TRAP EXPERIMENT ---
print("\n--- LEAKAGE TRAP EXPERIMENT ---")

# Define a realistic target label based on gsc_clicks activity
df['target_label'] = (df['gsc_clicks'] > 0).astype(int) if 'gsc_clicks' in df.columns else (df[col_word] > 0).astype(int)
y = df['target_label']

# 1. Honest Model Evaluation (using feature subset to show realistic baseline score)
honest_cols = [c for c in feature_cols if c != 'gsc_clicks']
if len(honest_cols) == 0:
    honest_cols = feature_cols

X_honest = df[honest_cols].fillna(0)
clf_honest = RandomForestClassifier(n_estimators=30, max_depth=3, random_state=42)
clf_honest.fit(X_honest, y)
honest_acc = accuracy_score(y, clf_honest.predict(X_honest))
print(f"1. Honest Model Accuracy: {honest_acc:.4f}")

# 2. Add Deliberate Leaked Feature (Derived directly from label)
df['leaked_future_signal'] = df['target_label'] * 0.98 + np.random.normal(0, 0.01, size=len(df))
X_leaked = df[honest_cols + ['leaked_future_signal']].fillna(0)

clf_leaked = RandomForestClassifier(n_estimators=30, max_depth=3, random_state=42)
clf_leaked.fit(X_leaked, y)
leaked_acc = accuracy_score(y, clf_leaked.predict(X_leaked))
print(f"2. Leaked Model Accuracy: {leaked_acc:.4f}  <-- Artificially inflated score!")

# 3. Clean Up & Revert
df.drop(columns=['leaked_future_signal', 'target_label'], inplace=True, errors='ignore')
print("3. Leaked feature removed. Honest feature set restored.")

--- Query 1: Grain Verification ---
 total_rows  unique_content_ids  duplicate_count
      30000               30000                0

--- Query 2: Slice Metrics ---
 total_records  avg_primary_metric
         30000               11.58

--- Query 3: Availability Verification (IS TRUE) ---
 total_slice_rows  valid_surviving_rows  survival_pct
            30000                  6810          22.7

--- LEAKAGE TRAP EXPERIMENT ---
1. Honest Model Accuracy: 0.9818
2. Leaked Model Accuracy: 1.0000  <-- Artificially inflated score!
3. Leaked feature removed. Honest feature set restored.


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

What This Data Can Never Tell You (Boundaries & Blindspots):

Off-Page & External Macro Signals: The dataset tracks on-page and search console metrics, but cannot observe competitor content overhauls, Google core algorithm updates, or industry search intent shifts.

GA4 vs. GSC Access Disparity: Certain clients in dim_clients have has_gsc_access = true but lack GA4 analytics integration (has_ga4_access = false). On-site engagement metrics (e.g., scroll rate, session duration) will be absent for GSC-only historical slices.

Non-Stationary Seasonality: A single mid-panel observation window (2026-03) cannot account for recurring annual traffic spikes or seasonal demand crashes.

In [16]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Evaluate missingness across selected feature matrix
print("Null count evaluation across configured feature fields:")
print(df[feature_cols].isnull().sum())

print("\nStatistical bounds check (showing distribution limits and variance):")
print(df[feature_cols].describe().T[['min', 'mean', '50%', 'max']])

Null count evaluation across configured feature fields:
gsc_impressions         0
gsc_clicks              0
gsc_sum_position        0
gsc_avg_position    23190
ga4_pageviews       10000
dtype: int64

Statistical bounds check (showing distribution limits and variance):
                  min       mean       50%      max
gsc_impressions   0.0    11.5833       0.0   7179.0
gsc_clicks        0.0   0.031467       0.0     15.0
gsc_sum_position  0.0  81.265433       0.0  41907.0
gsc_avg_position  0.0  12.896294  6.716298    139.0
ga4_pageviews     0.0     0.0016       0.0      2.0


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.